## Make Detection with the Trained Model

In [1]:
import mediapipe as mp
import cv2
import numpy as np
import pandas as pd

import pickle

import warnings
warnings.filterwarnings('ignore')

# Drawing helpers
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

### Reconstruct the input structure

In [2]:
# Determine important landmarks for warrior2
IMPORTANT_LMS = [
    "NOSE",             # 0
    "LEFT_EYE",         # 2
    "RIGHT_EYE",        # 5
    "LEFT_SHOULDER",    # 11
    "RIGHT_SHOULDER",   # 12
    "LEFT_ELBOW",       # 13
    "RIGHT_ELBOW",      # 14
    "LEFT_INDEX",       # 19
    "RIGHT_INDEX",      # 20
    "LEFT_HIP",         # 23
    "RIGHT_HIP",        # 24
    "LEFT_KNEE",        # 25
    "RIGHT_KNEE",       # 26
    "LEFT_HEEL",        # 29
    "RIGHT_HEEL",       # 30
    "LEFT_FOOT_INDEX",  # 31
    "RIGHT_FOOT_INDEX"  # 32
    

#     "RIGHT_SHOULDER",
# "LEFT_HEEL",
# "LEFT_INDEX",
# "NOSE",
# "LEFT_FOOT_INDEX",
# "RIGHT_HIP",
# "RIGHT_FOOT_INDEX",
# "LEFT_KNEE",
# "RIGHT_ELBOW",
# "RIGHT_EYE",
# "LEFT_SHOULDER",
# "RIGHT_KNEE",
# "LEFT_ELBOW",
# "RIGHT_HEEL",
# "LEFT_HIP",
# "LEFT_EYE",
# "RIGHT_INDEX"


]


# Generate all columns of the data frame

HEADERS = ["label"] # Label column

for lm in IMPORTANT_LMS:
    HEADERS += [f"{lm.lower()}_x", f"{lm.lower()}_y", f"{lm.lower()}_z", f"{lm.lower()}_v"]








    # HEADERS += [f"{lm.lower()}_x_scaled", f"{lm.lower()}_y_scaled", f"{lm.lower()}_z_scaled", f"{lm.lower()}_v"]

### Setup some important functions

In [3]:
def extract_important_keypoints(results) -> list:
    '''
    Extract important keypoints from mediapipe pose detection
    '''
    landmarks = results.pose_landmarks.landmark

    data = []
    for lm in IMPORTANT_LMS:
        keypoint = landmarks[mp_pose.PoseLandmark[lm].value]
        data.append([keypoint.x, keypoint.y, keypoint.z, keypoint.visibility])
    
    return np.array(data).flatten().tolist()


def rescale_frame(frame, percent=50):
    '''
    Rescale a frame to a certain percentage compare to its original frame
    '''
    width = int(frame.shape[1] * percent/ 100)
    height = int(frame.shape[0] * percent/ 100)
    dim = (width, height)
    return cv2.resize(frame, dim, interpolation =cv2.INTER_AREA)


# def rescale_frame(frame, max_dim=800):
#     """
#     Resize so the largest side is max_dim (keeps aspect ratio).
#     Returns original if already smaller.
#     """
#     h, w = frame.shape[:2]
#     if max(h, w) <= max_dim:
#         return frame
#     scale = max_dim / max(h, w)
#     dim = (int(w * scale), int(h * scale))
#     return cv2.resize(frame, dim, interpolation=cv2.INTER_AREA)


In [4]:

# VIDEO_TEST = "warrior-ii -1 (online-video-cutter.com).mp4"
# VIDEO_TEST = "Virabhadrasana ii -3.mp4"
VIDEO_TEST = "warrior-ii -1.mp4"

## 1. Make detection with Scikit learn model

In [5]:
# Load model
# with open("./model/LR_model.pkl", "rb") as f:
#     sklearn_model = pickle.load(f)

# # Load input scaler
# with open("./model/input_scaler.pkl", "rb") as f2:
#     input_scaler = pickle.load(f2)


with open("./model/warrior2_pipeline.pkl", "rb") as f:
    plank_pipeline = pickle.load(f)

# Transform prediction into class
def get_class(prediction: float) -> str:
    return {
    0:'c',
    1:'bent_front_knee',
    2:'arms_dropped',
    3:'arms_bent',
    4:'hips_rotated',
    5:'back_foot_wrong_angle',
    6:'leaning_forward',
    7:'narrow_stance',
    8:'shoulders_not_aligned'
    }.get(prediction)


In [17]:
import sys
import os

# Add the path to the parent folder of 'angle_calculation'
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(''), '../angle_calculation')))


In [18]:
from warrior_2_angle_calculation import (
    compute_elbow_angle,
    compute_shoulder_line_angle,
    compute_knee_angle,
    compute_hip_angle,
    compute_torso_angle
)

def compute_all_angles(row):
    """
    Compute all required angles for Warrior II pose.
    Returns a list of angles in the same order used during training.
    """

    angles = [
        # Arms
        compute_elbow_angle(row, side="left"),
        compute_elbow_angle(row, side="right"),

        # Shoulder horizontal alignment
        compute_shoulder_line_angle(row, side="left"),
        compute_shoulder_line_angle(row, side="right"),

        # Legs
        compute_knee_angle(row, side="left"),
        compute_knee_angle(row, side="right"),

        # Hips
        compute_hip_angle(row, side="left"),
        compute_hip_angle(row, side="right"),

        # Torso verticality
        compute_torso_angle(row, side="left"),
        compute_torso_angle(row, side="right"),
    ]

    return angles


In [19]:
# Angle column names (same order as compute_all_angles)
ANGLE_COLUMNS = [
        "left_elbow_angle", "right_elbow_angle",
        "left_shoulder_line_angle", "right_shoulder_line_angle",
        "left_knee_angle", "right_knee_angle",
        "left_hip_angle", "right_hip_angle",
        "left_torso_angle", "right_torso_angle"
]


In [ ]:
cap = cv2.VideoCapture(VIDEO_TEST)
current_stage = ""
prediction_probability_threshold = 0.80

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, image = cap.read()

        if not ret:
            break

        # Reduce size of a frame
        image = rescale_frame(image, 50)
        # image = cv2.flip(image, 1)

        # Recolor image from BGR to RGB for mediapipe
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False

        results = pose.process(image)

        if not results.pose_landmarks:
            print("No human found")
            continue

        # Recolor image from BGR to RGB for mediapipe
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # Draw landmarks and connections
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS, mp_drawing.DrawingSpec(color=(244, 117, 66), thickness=2, circle_radius=2), mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=1))

        # Make detection
        try:
            # Extract keypoints from frame for the input
            # row = extract_important_keypoints(results)

            # X_scaled = input_scaler.transform([row])  # [row] makes it 2D
    
            # # Make prediction and its probability
            # predicted_class = sklearn_model.predict(X_scaled)[0]
            # predicted_class = get_class(predicted_class)
            # prediction_probability = sklearn_model.predict_proba(X_scaled)[0]

            row = extract_important_keypoints(results)  # list of landmarks
            X_landmarks = pd.DataFrame([row], columns=HEADERS[1:])  # only landmarks columns

            # Compute all angles for this frame
            angles = compute_all_angles(X_landmarks.iloc[0])  # returns list of angles in the same order as training
            X_angles = pd.DataFrame([angles], columns=ANGLE_COLUMNS)  # ANGLE_COLUMNS = list of your angle column names

            # Combine landmarks + angles
            X_full = pd.concat([X_landmarks, X_angles], axis=1)

            # Predict with the pipeline (no need to scale manually)
            predicted_class = plank_pipeline.predict(X_full)[0]
            predicted_class = get_class(predicted_class)

            prediction_probability = plank_pipeline.predict_proba(X_full)[0]
            # print(predicted_class, prediction_probability)



            # X = pd.DataFrame([row], columns=HEADERS[1:])
            # X = pd.DataFrame(input_scaler.transform(X))

            # # Make prediction and its probability
            # predicted_class = sklearn_model.predict(X)[0]
            # predicted_class = get_class(predicted_class)
            # prediction_probability = sklearn_model.predict_proba(X)[0]


            # print(predicted_class, prediction_probability)




            # Evaluate model prediction
            # print(f"Predicted class: {predicted_class}")
            # if predicted_class == 0 and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
            #     current_stage = "Correct"
            # elif predicted_class == 1 and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
            #     current_stage = "Bent Front Knee"
            # elif predicted_class == 2 and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
            #     current_stage = "Arms Dropped"
            # elif predicted_class == 3 and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
            #     current_stage = "Arms Bent"
            # elif predicted_class == 4 and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
            #     current_stage = "Hips Rotated"
            # elif predicted_class == 5 and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
            #     current_stage = "Back Foot Wrong Angle"
            # elif predicted_class == 6 and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
            #     current_stage = "Leaning Forward"
            # elif predicted_class == 7 and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
            #     current_stage = "Narrow Stance"
            # elif predicted_class == 8 and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
            #     current_stage = "Shoulders Not Aligned"


            # print(f"Predicted class: {predicted_class}")
            if predicted_class == 'c' and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
                current_stage = "Correct"
            elif predicted_class == 'bent_front_knee' and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
                current_stage = "Bent Front Knee"
            elif predicted_class == 'arms_dropped' and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
                current_stage = "Arms Dropped"
            elif predicted_class == 'arms_bent' and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
                current_stage = "Arms Bent"
            elif predicted_class == 'hips_rotated' and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
                current_stage = "Hips Rotated"
            elif predicted_class == 'back_foot_wrong_angle' and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
                current_stage = "Back Foot Wrong Angle"
            elif predicted_class == 'leaning_forward' and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
                current_stage = "Leaning Forward"
            elif predicted_class == 'narrow_stance' and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
                current_stage = "Narrow Stance"
            elif predicted_class == 'shoulders_not_aligned' and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
                current_stage = "Shoulders Not Aligned"

            else:
                current_stage = "Unknown"

            
            # Visualization
            # Status box
            cv2.rectangle(image, (0, 0), (250, 60), (245, 117, 16), -1)

            # Display class
            cv2.putText(image, "CLASS", (95, 12), cv2.FONT_HERSHEY_COMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
            cv2.putText(image, current_stage, (90, 40), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

            # Display probability
            cv2.putText(image, "PROB", (15, 12), cv2.FONT_HERSHEY_COMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
            cv2.putText(image, str(round(prediction_probability[np.argmax(prediction_probability)], 2)), (10, 40), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

        except Exception as e:
            print(f"Error: {e}")
        
        cv2.imshow("CV2", image)
        
        # Press Q to close cv2 window
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

    

    for i in range (1, 5):
        cv2.waitKey(1)
  

arms_dropped [5.4623622e-05 7.4581505e-05 9.2704254e-01 7.1617872e-02 8.2979648e-05
 7.0496048e-06 5.9674482e-04 2.8853832e-05 4.9467804e-04]
arms_dropped [6.6807690e-05 6.5498898e-05 9.1138172e-01 8.7232135e-02 4.2423133e-05
 9.7764432e-06 7.2684826e-04 3.1574516e-05 4.4326088e-04]
arms_dropped [9.3779054e-05 7.1473893e-05 7.4455321e-01 2.5329491e-01 5.7399677e-05
 2.1616361e-05 1.3059410e-03 4.0477255e-05 5.6113640e-04]
arms_dropped [1.2117406e-04 7.7649900e-05 7.0922750e-01 2.8850377e-01 6.3448024e-05
 9.7524629e-05 7.7926082e-04 4.4043816e-05 1.0856468e-03]
arms_dropped [1.6809057e-04 8.6458473e-05 5.6278175e-01 4.3513471e-01 5.3116888e-05
 1.7311172e-04 2.3451250e-04 9.6249794e-05 1.2719868e-03]
arms_dropped [1.37876516e-04 1.01667836e-04 5.72937965e-01 4.24683243e-01
 7.87116078e-05 2.02629322e-04 2.04129072e-04 8.97668288e-05
 1.56400411e-03]
arms_dropped [1.4734783e-04 7.9025463e-05 7.8940207e-01 2.0897846e-01 6.4049709e-05
 1.3918812e-04 1.1894167e-04 6.4610940e-05 1.0063138e-

## 2. Make detection with Deep Learning Model

In [ ]:
# Load model
with open("./model/plank_dp.pkl", "rb") as f:
    deep_learning_model = pickle.load(f)

In [ ]:
cap = cv2.VideoCapture(VIDEO_TEST)
current_stage = ""
prediction_probability_threshold = 0.8

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, image = cap.read()

        if not ret:
            break

        # Reduce size of a frame
        image = rescale_frame(image, 50)

        # Recolor image from BGR to RGB for mediapipe
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False

        results = pose.process(image)

        if not results.pose_landmarks:
            print("No human found")
            continue

        # Recolor image from BGR to RGB for mediapipe
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # Draw landmarks and connections
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS, mp_drawing.DrawingSpec(color=(244, 117, 66), thickness=2, circle_radius=2), mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=1))

        # Make detection
        try:
            # Extract keypoints from frame for the input
            row = extract_important_keypoints(results)
            X = pd.DataFrame([row, ], columns=HEADERS[1:])
            X = pd.DataFrame(input_scaler.transform(X))
            

            # Make prediction and its probability
            prediction = deep_learning_model.predict(X)
            predicted_class = np.argmax(prediction, axis=1)[0]

            prediction_probability = max(prediction.tolist()[0])
            # print(X)

            # Evaluate model prediction
            if predicted_class == 0 and prediction_probability >= prediction_probability_threshold:
                current_stage = "Correct"
            elif predicted_class == 2 and prediction_probability >= prediction_probability_threshold: 
                current_stage = "Low back"
            elif predicted_class == 1 and prediction_probability >= prediction_probability_threshold: 
                current_stage = "High back"
            else:
                current_stage = "Unknown"

            # Visualization
            # Status box
            cv2.rectangle(image, (0, 0), (550, 60), (245, 117, 16), -1)

            # # Display class
            cv2.putText(image, "DETECTION", (95, 12), cv2.FONT_HERSHEY_COMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
            cv2.putText(image, current_stage, (90, 40), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

            # # Display class
            cv2.putText(image, "CLASS", (350, 12), cv2.FONT_HERSHEY_COMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
            cv2.putText(image, str(predicted_class), (345, 40), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

            # # Display probability
            cv2.putText(image, "PROB", (15, 12), cv2.FONT_HERSHEY_COMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
            cv2.putText(image, str(round(prediction_probability, 2)), (10, 40), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

        except Exception as e:
            print(f"Error: {e}")
        
        cv2.imshow("CV2", image)
        
        # Press Q to close cv2 window
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

    for i in range (1, 5):
        cv2.waitKey(1)
  

In [ ]:
X